# Student Lab Assignment — Issues 4–9

This notebook resolves the remaining data-quality issues in `clean_orders`, enriches the orders with `dim_products.csv`, calculates business metrics, and produces a profit pivot table by category and shipping city.

**Prerequisite:** `clean_orders` must already exist from the previous lab section, with Issues 1–3 resolved.

## Load Required Libraries and Inspect the Input

The code below imports pandas and performs a quick check of the supplied `clean_orders` DataFrame.

In [ ]:
import pandas as pd

# Confirm that clean_orders is available from the previous lab section.
display(clean_orders.head())
print(f"Rows: {len(clean_orders):,}")
print(f"Columns: {len(clean_orders.columns)}")

## Issue 4 — Negative Quantities

The `Units` column contains a negative value caused by a returns-entry error. Because the assignment explicitly requires quantities to be converted to their absolute values, `.abs()` is applied to the entire column.

In [ ]:
# Convert all Units values to their absolute values.
clean_orders["Units"] = clean_orders["Units"].abs()

print("Minimum Units after Issue 4 fix:", clean_orders["Units"].min())

## Issue 5 — Gross Outlier

ORD-113 contains `500` units, which is an obvious data-entry typo for `5`. For this lab, values greater than 20 are replaced with 5. This prevents the outlier from artificially inflating revenue and profit.

In [ ]:
# Replace the gross quantity outlier with the intended value of 5.
clean_orders.loc[clean_orders["Units"] > 20, "Units"] = 5

print("Maximum Units after Issue 5 fix:", clean_orders["Units"].max())

## Issue 6 — String Standardisation

- `Customer_Name`: trim whitespace and convert to Title Case.
- `Shipping_City`: trim whitespace and convert to Title Case.
- `Payment_Status`: trim whitespace and convert to uppercase.

This ensures equivalent values are represented consistently during grouping and analysis.

In [ ]:
clean_orders["Customer_Name"] = (
    clean_orders["Customer_Name"].str.strip().str.title()
)

clean_orders["Shipping_City"] = (
    clean_orders["Shipping_City"].str.strip().str.title()
)

clean_orders["Payment_Status"] = (
    clean_orders["Payment_Status"].str.strip().str.upper()
)

display(clean_orders[["Customer_Name", "Shipping_City", "Payment_Status"]].head())

## Issue 7 — Date Feature Engineering

`Order_Date` is converted to datetime and three analytical features are created:
- `Order_Month`: numeric month.
- `Order_DayName`: day name such as Monday or Tuesday.
- `Is_Weekend`: `True` for Saturday/Sunday and `False` otherwise.

In [ ]:
clean_orders["Order_Date"] = pd.to_datetime(clean_orders["Order_Date"])

clean_orders["Order_Month"] = clean_orders["Order_Date"].dt.month
clean_orders["Order_DayName"] = clean_orders["Order_Date"].dt.day_name()
clean_orders["Is_Weekend"] = clean_orders["Order_Date"].dt.dayofweek >= 5

display(
    clean_orders[
        ["Order_Date", "Order_Month", "Order_DayName", "Is_Weekend"]
    ].head()
)

## Issue 8 — Relational Enrichment

`dim_products.csv` is loaded and joined to `clean_orders` using a LEFT JOIN on `Product_SKU`. A LEFT JOIN preserves every order even when a matching product record is unavailable.

Business metrics:
- **Gross Revenue** = `Units × Unit_Price`
- **Net Profit** = `(Unit_Price − Cost_Price) × Units`

In [ ]:
# Load the product dimension table.
dim_products = pd.read_csv("dim_products.csv")

# Enrich orders while retaining every order record.
final_analytical_df = clean_orders.merge(
    dim_products,
    on="Product_SKU",
    how="left"
)

# Calculate business metrics.
final_analytical_df["Gross_Revenue"] = (
    final_analytical_df["Units"] * final_analytical_df["Unit_Price"]
)

final_analytical_df["Net_Profit"] = (
    (final_analytical_df["Unit_Price"] - final_analytical_df["Cost_Price"])
    * final_analytical_df["Units"]
)

display(final_analytical_df.head())

### Enrichment Validation

Check that the final DataFrame has retained the expected orders and that the new product and financial columns are present.

In [ ]:
print(f"Final rows: {len(final_analytical_df):,}")
print("Final columns:")
print(final_analytical_df.columns.tolist())

print("\nMissing Cost_Price values:", final_analytical_df["Cost_Price"].isna().sum())
print("Missing Category values:", final_analytical_df["Category"].isna().sum())

## Issue 9 — Cross-Tabulation Analysis

The pivot table summarizes total `Net_Profit` by `Category` (rows) and `Shipping_City` (columns). `margins=True` adds row and column totals.

In [ ]:
profit_pivot = pd.pivot_table(
    final_analytical_df,
    values="Net_Profit",
    index="Category",
    columns="Shipping_City",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
)

display(profit_pivot)

## Final Quality Checks

These checks confirm that the key transformations were applied successfully.

In [ ]:
# Verify that quantities are non-negative and no gross outlier remains.
assert (final_analytical_df["Units"] >= 0).all()
assert final_analytical_df["Units"].max() <= 20

# Verify that the required analytical columns exist.
required_columns = [
    "Order_Month", "Order_DayName", "Is_Weekend",
    "Gross_Revenue", "Net_Profit"
]
missing_columns = [c for c in required_columns if c not in final_analytical_df.columns]
assert not missing_columns, f"Missing columns: {missing_columns}"

print("All final quality checks passed.")
display(final_analytical_df)

## Submission Notes

`final_analytical_df` is the required analysis-ready DataFrame and `profit_pivot` contains the required Issue 9 aggregation.

After saving this notebook in your Git repository, commit and push it with:

```bash
git add student_lab_issues_4_9.ipynb
git commit -m "Complete student lab issues 4-9"
git push origin main
```

If your repository uses a different branch, replace `main` with that branch name.